# Moon Rover MPC Testing

In [ ]:
%matplotlib ipympl

import time
import functools
import jax
import jax.numpy as jnp
import numpy as np
import tqdm
import scipy.optimize as sci_opt
import pandas as pd

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.viz as viz
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.quartic_cost as quartic_cost

jax.config.update("jax_enable_x64", True)


In [ ]:
def load_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    """Load reference linear accelerations and angular velocities."""
    df = pd.read_csv(file_path)

    acc_keys = [f"sesmt.md.merged_frame.xyz_acc[{i}] {{m/s^2}}" for i in range(3)]
    omega_keys = [f"sesmt.md.merged_frame.ang_vel[{i}] {{rad/s}}" for i in range(3)]
    gravity_keys = [
        f"sesmt.md.merged_frame.gravity[{i}] {{m/s^2}}" for i in range(3)
    ]

    acc_ref = jnp.array(df[acc_keys])
    omega_ref = jnp.array(df[omega_keys])
    gravity_ref = jnp.array(df[gravity_keys])

    # for some reason, data collection after a lot of nonsense data
    # we grab the data after we start recognizing nonzero (x) accelerations
    # note that using direct equality is desired here (and not jnp.isclose)
    offset = jnp.argmax(acc_ref[:, 0] != 0.0)

    # we need to cancel and then add back in the gravity vector
    acc_ref = acc_ref[offset:, :] - 2 * gravity_ref[offset:, :]
    omega_ref = omega_ref[offset:, :]
    
    return acc_ref, omega_ref


In [ ]:
# load data
file_path = "/Users/jozbee/work/eng/comp/data/00_sms_drive.csv"
acc_ref, omega_ref = load_sms_references(file_path)
assert acc_ref.shape[0] == omega_ref.shape[0]
assert acc_ref.shape[1] == 3
assert omega_ref.shape[1] == 3

In [ ]:
# general setup
# num_steps = acc_ref.shape[0]
begin = 1000
# num_steps = acc_ref.shape[0] - begin
num_steps = 1000
n = 400  # horizon

In [ ]:
# cost setup

weights = opt.Weights(
    acc=jnp.array([1e2, 1e2, 1e0]),
    omega=jnp.array([1e4, 1e4, 1e4]),
    alpha_acc=jnp.array([4.0]),
    alpha_omega=jnp.array([4.0]),
)

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    # sizes=[1.0, 2**3, 2**5, 2**10],
    sizes=[1.0, 2**5, 2**8, 2**11],
    low=const.leg_min,
    high=const.leg_max,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_leg_vel,
    high=const.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_yaw,
    high=const.max_yaw,
)


In [ ]:
# train setup
acados_get_cost = functools.partial(
    opt.get_cost,
    weights=weights,
    # acc_ref=acc_ref,
    # omega_ref=omega_ref,
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    yaw_cost=yaw_cost,
    # state0=state0,
)


def train_step(
    state0: jax.Array,
    last_control: jax.Array,
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    **kwargs,
) -> tuple[jax.Array, jax.Array, utils.TableSol, sci_opt.OptimizeResult, float]:
    acc_ref = jnp.ravel(acc_ref)
    omega_ref = jnp.ravel(omega_ref)
    assert acc_ref.size == 3
    assert omega_ref.size == 3

    acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
    omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

    cost, cost_jac, _ = acados_get_cost(
        acc_ref=acc_ref, omega_ref=omega_ref, state0=state0
    )
    t0 = time.time()
    opt_options = kwargs.get("opt_options", {"maxiter": 16, "maxls": 8})
    res = sci_opt.minimize(
        fun=cost,
        x0=np.array(last_control),
        method="L-BFGS-B",
        jac=cost_jac,
        options=opt_options,
    )
    t1 = time.time()
    t_tot = t1 - t0
    control = opt.Control.from_flat(jnp.array(res.x))
    state = control.get_state(state0)
    table_sol = utils.TableSol(
        x=jnp.column_stack(list(state)),
        u=jnp.column_stack(list(control)),
        stats=utils.TableStats(
            time=jnp.squeeze(t_tot),
            status=jnp.squeeze(res.status),
            cost=jnp.squeeze(res.fun),
        ),
    )
    return state[1].flatten(), control.flatten(), table_sol, res, t_tot


In [ ]:
# run
state0 = jnp.zeros(12, dtype=float)
times = []
sol_list = []
res_list = []

# pre-compile, and initialize last_control
_, last_control, _, _, _ = train_step(
    state0, jnp.zeros(6 * n, dtype=float), acc_ref[begin + 0], omega_ref[begin + 0]
)

for i in tqdm.tqdm(range(num_steps)):
    state0, last_control, sol, res, t_tot = train_step(
        state0, last_control, acc_ref[begin + i], omega_ref[begin + i], opt_options={"maxiter": 8, "maxls": 4}
    )
    sol_list.append(sol)
    res_list.append(res)
    times.append(t_tot)


In [ ]:
freqs = 1.0 / np.array(times)
float(np.min(freqs)), float(np.max(freqs)), float(np.mean(freqs)), float(np.std(freqs))

In [ ]:
plt.close("all")

In [ ]:
anim_fig = viz.animate_trajectory(
    trajectory=sol_list,
    sim_rate=1.0,
    fps=30,
)

In [ ]:
# anim_fig[0].save("/Users/jozbee/work/eng/comp/data/flipped_table.mp4", fps=30)

In [ ]:
sol_list_end = []
extra_steps = 0

# sol_list_end = viz.split_tablesol(sol_list[-1])
# extra_steps = sol_list[-1].u.shape[0] - 1

In [ ]:
acados_human_fig = viz.plot_human_trajectory(
    trajectory=sol_list + sol_list_end,
    references={
        "xyz-acceleration": np.array(acc_ref[begin: begin + num_steps]),
        "angular-velocity": np.array(omega_ref[begin: begin + num_steps]),
    },
)

In [ ]:
acados_table_fig = viz.plot_cartesian_table_trajectory(
    trajectory=sol_list + sol_list_end,
)

In [ ]:
acados_actuator_fig = viz.plot_actuator_trajectory(
    trajectory=sol_list + sol_list_end,
)